# TFM - DenseNet201 Escenario 1 (HAM10000) en Google Colab

Notebook adaptado del script `densenet201_ham10000_escenario1.py` para correr con GPU gratuita de Colab.

**Antes de correr:**
1. Activa GPU: `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → `GPU` (T4 es suficiente).
2. Necesitas tu archivo `kaggle.json` (Kaggle → Account → API → Create New Token) para descargar el dataset HAM10000.

**Qué hace cada celda, en orden:**
1. Monta Google Drive (ahí se guardan resultados y checkpoints, para que sobrevivan si Colab te desconecta).
2. Instala dependencias que no vienen por defecto.
3. Configura tus credenciales de Kaggle.
4. Descarga y descomprime el dataset HAM10000 (~3 GB, al disco local de la sesión, rápido pero temporal).
5. Verifica que la GPU esté disponible.
6. Corre el script completo (arquitectura, entrenamiento con checkpoints, evaluación y generación del Excel).

Si Colab te desconecta a mitad de entrenamiento: vuelve a correr las celdas 1, 4 y 5 (Drive, dataset, GPU) y luego la celda 6 de nuevo — el script detecta el checkpoint guardado en Drive y continúa desde la última época completada, no desde cero.


## 1. Montar Google Drive (para resultados y checkpoints persistentes)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/TFM/Resultados', exist_ok=True)
print("Carpeta de resultados lista en /content/drive/MyDrive/TFM/Resultados")


## 2. Instalar dependencias que no vienen preinstaladas en Colab

In [ ]:
!pip install -q kaggle openpyxl


## 3. Configurar credenciales de Kaggle

Kaggle tiene dos formas distintas de mostrarte el token, dependiendo de tu cuenta:

- **Si te descargó un archivo `kaggle.json`**: ejecuta la celda y cuando te lo pida, súbelo.
- **Si en vez de un archivo te mostró solo el texto del API key** (un string largo de letras/números, junto a tu nombre de usuario): elige esa opción en la celda y pega tu **username** de Kaggle (el que aparece en tu perfil, no tu email) y el **key** que copiaste.

En ambos casos termina guardado correctamente en `/root/.kaggle/kaggle.json`.

In [ ]:
import os, json, shutil
from getpass import getpass

os.makedirs('/root/.kaggle', exist_ok=True)

modo = input("¿Tienes el archivo kaggle.json (J) o solo el texto del username/key (T)? [J/T]: ").strip().upper()

if modo == "J":
    from google.colab import files
    print("Sube tu archivo kaggle.json:")
    uploaded = files.upload()
    nombre_subido = list(uploaded.keys())[0]
    shutil.move(nombre_subido, '/root/.kaggle/kaggle.json')
else:
    kaggle_username = input("Tu username de Kaggle (el de tu perfil, NO el email): ").strip()
    kaggle_key = getpass("Tu Kaggle API key (no se mostrará mientras escribes): ").strip()
    with open('/root/.kaggle/kaggle.json', 'w') as f:
        json.dump({"username": kaggle_username, "key": kaggle_key}, f)

os.chmod('/root/.kaggle/kaggle.json', 0o600)

# Verificación rápida sin descargar nada todavía
with open('/root/.kaggle/kaggle.json') as f:
    contenido = json.load(f)
print(f"Credenciales guardadas para el usuario: {contenido['username']}")


¿Tienes el archivo kaggle.json (J) o solo el texto del username/key (T)? [J/T]: T
Tu username de Kaggle (el de tu perfil, NO el email): marzo27
Tu Kaggle API key (no se mostrará mientras escribes): ··········
Credenciales guardadas para el usuario: marzo27


## 4. Descargar y descomprimir el dataset HAM10000

In [ ]:
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p /content/_ham10000_zip
!mkdir -p /content/HAM10000
!unzip -q -o /content/_ham10000_zip/skin-cancer-mnist-ham10000.zip -d /content/HAM10000
!rm -rf /content/_ham10000_zip
print("Dataset descomprimido en /content/HAM10000")
!find /content/HAM10000 -maxdepth 2 -type d


Dataset URL: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000
License(s): CC-BY-NC-SA-4.0
100% 5.20G/5.20G [00:43<00:00, 129MB/s]

Dataset descomprimido en /content/HAM10000
/content/HAM10000
/content/HAM10000/ham10000_images_part_2
/content/HAM10000/HAM10000_images_part_2
/content/HAM10000/HAM10000_images_part_1
/content/HAM10000/ham10000_images_part_1


In [ ]:
import os
import shutil

origen = '/content/HAM10000/ham10000_images_part_1/'
destino = '/content/HAM10000/HAM10000_images_part_1/'


for archivo in os.listdir(origen):
    ruta_f_origen = os.path.join(origen, archivo)
    ruta_f_destino = os.path.join(destino, archivo)

    # Filtra para copiar únicamente archivos, omitiendo subcarpetas
    if os.path.isfile(ruta_f_origen):
        shutil.copy2(ruta_f_origen, ruta_f_destino)

shutil.rmtree(origen)

archivos = [f for f in os.listdir(destino) if os.path.isfile(os.path.join(destino, f))]
cantidad = len(archivos)
print(f"La carpeta destino tiene un total de {cantidad} archivos.")

print("Copiado selectivo completado. Parte 1.")

origen = '/content/HAM10000/ham10000_images_part_2/'
destino = '/content/HAM10000/HAM10000_images_part_2/'


for archivo in os.listdir(origen):
    ruta_f_origen = os.path.join(origen, archivo)
    ruta_f_destino = os.path.join(destino, archivo)

    # Filtra para copiar únicamente archivos, omitiendo subcarpetas
    if os.path.isfile(ruta_f_origen):
        shutil.copy2(ruta_f_origen, ruta_f_destino)

shutil.rmtree(origen)

archivos = [f for f in os.listdir(destino) if os.path.isfile(os.path.join(destino, f))]
cantidad = len(archivos)
print(f"La carpeta destino tiene un total de {cantidad} archivos.")

print("Copiado selectivo completado. Parte 2.")

## 5. Verificar que la GPU esté disponible

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("AVISO: no hay GPU asignada. Ve a 'Entorno de ejecución' -> "
          "'Cambiar tipo de entorno de ejecución' -> selecciona GPU, "
          "y vuelve a correr el notebook desde la celda 1.")


PyTorch: 2.11.0+cu128
CUDA disponible: True
GPU: Tesla T4


## 6. Script completo (arquitectura, entrenamiento, evaluación, Excel)

Mismo código que `densenet201_ham10000_escenario1.py`, solo con las rutas adaptadas a Colab
(dataset en `/content/HAM10000`, resultados y checkpoints en Google Drive).

Para cambiar el experimento (`plano_sin_da`, `plano_con_da`, `multitarea_fijo`,
`multitarea_curriculum`), edita la línea `EXPERIMENTO = "multitarea_curriculum"` más abajo
antes de correr la celda.

In [ ]:
# -*- coding: utf-8 -*-
"""
densenet201_ham10000_escenario1.py

Reproduce el modelo DenseNet201 - Escenario 1 (dataset multiclase original,
7 clases) descrito en las secciones 5.4.4 y 5.5.4 del borrador del TFM
(TFM_Entrega3_v6.docx), y genera un Excel de métricas comparable al de SVM
(metricas.xlsx, 7 hojas, sección 5.6.1.1) para alimentar el análisis
comparativo de la sección 5.5.6.3.

==============================================================================
TRAZABILIDAD DE CADA PARÁMETRO (para que el autor pueda auditar el script
línea por línea sin que se le presenten supuestos como si fueran hechos
del TFM):

    [TFM]   Tomado literalmente de las secciones 5.4.1 / 5.4.4 / 5.5.4 /
            5.5.6 del borrador TFM_Entrega3_v6.docx.
    [NB]    El TFM (5.4.4/5.5.4) NO especifica este dato. Se usó como
            referencia ÚNICA Y EXCLUSIVAMENTE TFM_1_v4.ipynb, tal como
            pediste, para estos 3 puntos concretos:
              1) tamaño de imagen de entrada
              2) batch size
              3) tipo concreto de "planificador dinámico" (scheduler)
    [SUP]   Detalle de implementación que ni el TFM ni el notebook
            especifican de forma inequívoca, necesario para que el script
            sea ejecutable. Se documenta la decisión tomada y el motivo,
            para que la confirmes o la corrijas tú mismo.

NO SE INVENTA NINGÚN RESULTADO: este script entrena y mide; no rellena
métricas con valores supuestos. Las métricas que aparecen como constantes
en SVM_ESCENARIO1_REFERENCIA (más abajo) son citas textuales de las Tablas
8, 9 y 10 del propio TFM_Entrega3_v6.docx, usadas solo para construir la
hoja de comparación.

------------------------------------------------------------------------------
RESUMEN DE LO QUE EL TFM SÍ DESCRIBE (5.5.4) Y QUE ESTE SCRIPT REPRODUCE
COMO UNA PROGRESIÓN DE 4 EXPERIMENTOS (selecciona uno con la variable
EXPERIMENTO más abajo):

  1) "plano_sin_da"
     Cabeza única de 7 neuronas (Softmax), Weighted Cross-Entropy Loss,
     SIN Data Augmentation, 30 épocas.
     Resultado reportado en el TFM: Accuracy 74.00%, Recall macro 0.68.

  2) "plano_con_da"
     Igual que (1) pero CON Data Augmentation (rotación ±20°, flips
     horizontal/vertical, RandomResizedCrop), 50 épocas, Adam lr=1e-4.
     Resultado reportado en el TFM: Accuracy 67.30% (empeora), Recall
     macro 0.68, Precision macro 0.46.

  3) "multitarea_fijo"
     Arquitectura bifurcada en dos cabezas lineales paralelas sobre el
     mismo backbone DenseNet201:
       - Cabeza A: binaria (Benigno = nv,bkl,df,vasc / Maligno = mel,bcc,akiec)
       - Cabeza B: específica (7 clases)
     Pérdida conjunta con peso FIJO 50/50, Data Augmentation activo,
     30 épocas, Adam + "planificador dinámico".
     Resultado reportado en el TFM: Accuracy 89.07%, Precision macro 0.80,
     F1 macro 0.83 (mejor época: 24).

  4) "multitarea_curriculum"  <-- EXPERIMENTO C1, valor por defecto
     Igual que (3) pero sustituyendo el peso fijo 50/50 por una función
     ESCALÓN (Curriculum Learning) sobre el hiperparámetro alpha:
       alpha = 0.3 en épocas 1-15 (fase de "anclaje" de rasgos binarios)
       alpha = 0.7 en épocas 16-30 (fase de "especialización" en 7 clases)
     Este es el resultado que el TFM reporta como "Escenario 1" en la
     Tabla de comparación 5.5.6.2 y en las Conclusiones: Accuracy 90.16%,
     Recall mel 75.3% (168/223), Recall nv 93.8% (1258/1341), sobre un
     test set de N=2003 imágenes.

     [SUP] El TFM da los DOS valores de alpha y los rangos de época, pero
     no especifica a cuál de las dos pérdidas se aplica cada peso. Se
     interpreta, por ser la lectura más consistente con el texto ("fase
     inicial de anclaje de características BINARIAS" = más peso a la
     tarea binaria; "fase de especialización EN 7 CLASES" = más peso a la
     tarea específica), que:
         loss_total = (1 - alpha) * loss_binaria + alpha * loss_especifica
     Si en el experimento original la asignación fue la inversa, edita la
     función get_alpha_pesos() más abajo.

------------------------------------------------------------------------------
[SUP] División train/test: el TFM no detalla un método de partición
específico dentro de 5.4.4. Se usa train_test_split estratificado por
clase 'dx', 80/20, random_state=42 -- igual que el script SVM de línea
base (sección 5.4.2, "[3/9] División train/test", 8.012 train / 2.003
test) y exactamente igual a TFM_1_v4.ipynb. Esto además es coherente con
la Tabla de la sección 5.5.6.2, donde "Escenario 1 (C1)" se reporta sobre
N=2003, el mismo tamaño del test set del SVM de línea base. La partición
agrupada por lesion_id (GroupShuffleSplit) es exclusiva del Escenario 2
(dataset balanceado) según el propio TFM -- no se aplica aquí.
==============================================================================
"""

import os
import json
import time
import copy
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_recall_fscore_support, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    cohen_kappa_score, matthews_corrcoef, confusion_matrix,
)
from sklearn.preprocessing import label_binarize

import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

# ==============================================================================
# 0. CONFIGURACIÓN GENERAL
# ==============================================================================

# [SUP] Ruta raíz actualizada según indicación del autor (antes D:\claude).
# ------------------------------------------------------------------------------
# [ADAPTADO PARA GOOGLE COLAB]
# El dataset se descarga en el disco local de la sesion de Colab (/content),
# que es rapido pero se BORRA si la sesion se desconecta. Los resultados y
# los checkpoints se guardan en Google Drive (montado en /content/drive),
# que SI persiste entre sesiones -- asi, si Colab te desconecta a mitad de
# entrenamiento, al reconectar y volver a correr el notebook el script
# encuentra el checkpoint en Drive y continua donde se quedo.
# ------------------------------------------------------------------------------
RUTA_BASE = "/content"
RUTA_DATOS = "/content/HAM10000"
RUTA_METADATA = os.path.join(RUTA_DATOS, "HAM10000_metadata.csv")
RUTA_IMG_PART1 = os.path.join(RUTA_DATOS, "HAM10000_images_part_1")
RUTA_IMG_PART2 = os.path.join(RUTA_DATOS, "HAM10000_images_part_2")
RUTA_RESULTADOS = "/content/drive/MyDrive/TFM/Resultados"
os.makedirs(RUTA_RESULTADOS, exist_ok=True)

# [TFM] 5.5.4 - mapeo de clases. Se usa el mismo MAPEO_FIJO_CLASES de
# TFM_1_v4.ipynb (orden alfabético) para que los índices numéricos sean
# coherentes si se cargan pesos guardados de una corrida previa.
MAPEO_FIJO_CLASES = {
    'akiec': 0,  # Queratosis actínica
    'bcc': 1,    # Carcinoma basocelular
    'bkl': 2,    # Lesión benigna tipo queratosis
    'df': 3,     # Dermatofibroma
    'mel': 4,    # Melanoma (maligno crítico)
    'nv': 5,     # Nevus melanocítico (lunar común)
    'vasc': 6,   # Lesión vascular
}
NOMBRES_CLASES = list(MAPEO_FIJO_CLASES.keys())
NUM_CLASES = len(NOMBRES_CLASES)

# [TFM] 5.5.4 - agrupación clínica usada por la Cabeza A (triaje binario):
# "Benigna (agrupando nv, bkl, df, vasc) o Maligna/Sospechosa (mel, bcc, akiec)"
CLASES_MALIGNAS = ['akiec', 'bcc', 'mel']
CLASES_BENIGNAS = ['bkl', 'df', 'nv', 'vasc']
IDX_MALIGNAS = [MAPEO_FIJO_CLASES[c] for c in CLASES_MALIGNAS]

# [NB] No especificado en el TFM para DenseNet201 (sí lo está para SVM: 64x64).
IMG_SIZE = (224, 224)

# [NB] No especificado en el TFM.
BATCH_SIZE = 32

# [TFM] 5.4.1 "misma semilla aleatoria (42)" -- aplicado de forma consistente
# en todo el TFM (SVM y, por extensión, aquí).
RANDOM_STATE = 42

# ------------------------------------------------------------------------------
# Selección del experimento a ejecutar (ver docstring superior).
# Cambia este valor para reproducir cualquiera de los 4 intentos narrados
# en 5.5.4. Por defecto se ejecuta el resultado final ("Escenario 1" /
# Experimento C1) que es el que aparece en las tablas comparativas del TFM.
# ------------------------------------------------------------------------------
EXPERIMENTO = "multitarea_curriculum"  # "plano_sin_da" | "plano_con_da" | "multitarea_fijo" | "multitarea_curriculum"

EPOCAS_POR_EXPERIMENTO = {
    "plano_sin_da": 30,           # [TFM]
    "plano_con_da": 50,           # [TFM]
    "multitarea_fijo": 30,        # [TFM]
    "multitarea_curriculum": 30,  # [TFM]
}

# [TFM] "el optimizador Adam con una tasa de aprendizaje inicial de eta=0.0001"
# (mencionado explícitamente en el intento 2; se mantiene en los intentos
# posteriores porque el TFM no vuelve a indicar un valor distinto).
OPTIMIZER_LR = 1e-4

# [NB] weight_decay no se menciona en el TFM; tomado de TFM_1_v4.ipynb.
WEIGHT_DECAY = 1e-4

# [NB] "planificador dinámico" (5.5.4) no se detalla en el TFM. En
# TFM_1_v4.ipynb el scheduler implementado es ReduceLROnPlateau.
SCHEDULER_FACTOR = 0.1
SCHEDULER_PATIENCE = 3

# [TFM] Data Augmentation descrito textualmente en el intento 2 de 5.5.4:
# "Rotaciones aleatorias en un rango de 20 grados", "Giros simétricos
# horizontales y verticales", "Recortes con ajuste de escala variable
# (Random Resized Crop)". Se mantiene activo en los intentos 3 y 4 según
# el propio texto ("manteniendo activo el Data Augmentation").
TRANSFORM_TRAIN = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
TRANSFORM_EVAL = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
# El intento 1 ("plano_sin_da") no usa aumento de datos, ni en train ni en val.
TRANSFORM_SIN_DA = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==============================================================================
# REFERENCIA SVM ESCENARIO 1 (para la hoja de comparación del Excel)
# Citado literalmente de TFM_Entrega3_v6.docx, Tabla 9 y Tabla 10
# (sección 5.5.2.2). No se recalcula nada: son los valores ya publicados
# en el propio borrador, copiados aquí solo para no tener que transcribirlos
# manualmente a Excel.
# ==============================================================================
SVM_ESCENARIO1_GLOBAL = {
    "accuracy": 0.6815, "balanced_accuracy": 0.5167,
    "f1_macro": 0.4834, "f1_weighted": 0.6990, "f1_micro": 0.6815,
    "precision_macro": 0.4668, "precision_weighted": 0.7292,
    "recall_macro": 0.5167, "recall_weighted": 0.6815,
    "auc_macro": 0.8776, "auc_weighted": 0.8726,
    "cohen_kappa": 0.4496, "mcc": 0.4561,
}
SVM_ESCENARIO1_POR_CLASE = {
    # clase: (soporte, precision, recall, f1, auc, avg_precision)
    "akiec": (65, 0.417, 0.662, 0.512, 0.932, 0.417),
    "bcc":   (103, 0.452, 0.553, 0.498, 0.907, 0.545),
    "bkl":   (220, 0.449, 0.545, 0.493, 0.851, 0.477),
    "df":    (23, 0.286, 0.261, 0.273, 0.872, 0.202),
    "mel":   (223, 0.338, 0.475, 0.395, 0.827, 0.335),
    "nv":    (1341, 0.890, 0.763, 0.822, 0.878, 0.936),
    "vasc":  (28, 0.435, 0.357, 0.392, 0.876, 0.196),
}
# [TFM] Tabla 13 - tiempos del SVM de línea base, citados para la hoja
# de costo computacional.
SVM_ESCENARIO1_TIEMPOS = {"train_s": 166.727, "infer_total_s": 6.731, "infer_per_img_ms": 3.360}


# ==============================================================================
# 1. DATASET
# ==============================================================================

class HAM10000Dataset(Dataset):
    """
    Dataset de PyTorch para HAM10000. Busca cada imagen en
    HAM10000_images_part_1 y _part_2 (estructura estándar de Kaggle,
    confirmada en TFM_Entrega3_v6.docx sección 5.6.3 / README del repo).
    """

    def __init__(self, dataframe, image_path_mapping, transform):
        self.df = dataframe.reset_index(drop=True)
        self.image_path_mapping = image_path_mapping
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = row["image_id"]
        label = MAPEO_FIJO_CLASES[row["dx"]]
        img_path = self.image_path_mapping[img_id]
        imagen = Image.open(img_path).convert("RGB")
        imagen = self.transform(imagen)
        return imagen, label


EXTENSIONES_IMAGEN = (".jpg", ".jpeg")


def construir_mapeo_imagenes():
    """
    Construye {image_id: ruta_completa} buscando imágenes .jpg/.jpeg.

    [SUP] La versión anterior solo miraba dentro de RUTA_IMG_PART1 /
    RUTA_IMG_PART2 (un nivel, sin recursividad), lo que falló si la
    estructura real de carpetas es distinta (subcarpetas anidadas, un solo
    folder, nombres distintos, etc.). Esta versión busca de forma
    RECURSIVA bajo RUTA_DATOS, igual que describe el TFM (5.4.2, "[1/9]
    Detección automática del dataset: busca recursivamente el CSV de
    metadatos y las carpetas de imágenes dentro de D:\\TFM\\HAM10000,
    siendo compatible con la estructura de una o dos partes de Kaggle").
    """
    mapeo = {}
    if not os.path.isdir(RUTA_DATOS):
        return mapeo
    for carpeta_actual, _subdirs, filenames in os.walk(RUTA_DATOS):
        for filename in filenames:
            if filename.lower().endswith(EXTENSIONES_IMAGEN):
                img_id = os.path.splitext(filename)[0]
                mapeo[img_id] = os.path.join(carpeta_actual, filename)
    return mapeo


def localizar_metadata():
    """
    Devuelve la ruta a HAM10000_metadata.csv. Si no está exactamente en
    RUTA_METADATA, busca recursivamente bajo RUTA_DATOS (y, como último
    recurso, bajo RUTA_BASE) antes de rendirse.
    """
    if os.path.isfile(RUTA_METADATA):
        return RUTA_METADATA
    for raiz in (RUTA_DATOS, RUTA_BASE):
        if os.path.isdir(raiz):
            for carpeta_actual, _subdirs, filenames in os.walk(raiz):
                for filename in filenames:
                    if filename.lower() == "ham10000_metadata.csv":
                        return os.path.join(carpeta_actual, filename)
    return None


def cargar_metadata_y_split():
    """
    Carga HAM10000_metadata.csv y realiza la división 80/20 estratificada
    por 'dx' con random_state=42. [SUP] ver justificación en el docstring
    superior del módulo.

    Antes de intentar el split, valida que realmente haya imágenes
    emparejadas y muestra diagnóstico detallado si no las hay, en vez de
    dejar que falle más adelante con un ValueError genérico de sklearn.
    """
    ruta_csv = localizar_metadata()
    if ruta_csv is None:
        raise FileNotFoundError(
            f"No se encontró HAM10000_metadata.csv ni en {RUTA_METADATA} "
            f"ni en ninguna subcarpeta de {RUTA_DATOS} o {RUTA_BASE}.\n"
            f"Revisa RUTA_BASE / RUTA_DATOS al inicio del script."
        )
    if ruta_csv != RUTA_METADATA:
        print(f"AVISO: HAM10000_metadata.csv no estaba en la ruta esperada "
              f"({RUTA_METADATA}). Se usó en su lugar: {ruta_csv}")
    df = pd.read_csv(ruta_csv)

    mapeo_imagenes = construir_mapeo_imagenes()

    if len(mapeo_imagenes) == 0:
        raise FileNotFoundError(
            f"No se encontró NINGUNA imagen (.jpg/.jpeg) bajo {RUTA_DATOS} "
            f"(búsqueda recursiva). El CSV de metadatos sí se cargó "
            f"({len(df)} filas), pero no hay imágenes con las que emparejarlo.\n"
            f"Verifica que las carpetas HAM10000_images_part_1 / "
            f"HAM10000_images_part_2 (o como se llamen en tu copia del "
            f"dataset) estén DENTRO de {RUTA_DATOS}, y que contengan "
            f"archivos .jpg directamente o en subcarpetas."
        )

    faltantes = set(df["image_id"]) - set(mapeo_imagenes.keys())
    cobertura = 1 - (len(faltantes) / len(df)) if len(df) else 0
    print(f"Imágenes encontradas en disco (búsqueda recursiva bajo "
          f"{RUTA_DATOS}): {len(mapeo_imagenes)}. "
          f"Filas del CSV emparejadas con una imagen: {len(df) - len(faltantes)} "
          f"de {len(df)} ({cobertura * 100:.1f}%).")

    if cobertura < 0.5:
        raise FileNotFoundError(
            f"Solo se pudo emparejar el {cobertura * 100:.1f}% de las filas "
            f"del CSV con una imagen en disco. Esto normalmente indica que "
            f"RUTA_DATOS no apunta a la carpeta correcta, o que el dataset "
            f"está incompleto. Revisa la estructura de carpetas antes de "
            f"continuar; el entrenamiento no se inicia con cobertura tan baja."
        )

    if faltantes:
        print(f"AVISO: {len(faltantes)} imágenes listadas en el CSV no se "
              f"encontraron en disco. Se excluyen del dataset.")
        df = df[~df["image_id"].isin(faltantes)].reset_index(drop=True)

    if len(df) == 0:
        raise ValueError("El dataset quedó vacío tras filtrar imágenes "
                          "faltantes. Revisa la correspondencia entre el "
                          "CSV y las carpetas de imágenes.")

    df_train, df_test = train_test_split(
        df, test_size=0.20, random_state=RANDOM_STATE, stratify=df["dx"]
    )
    print(f"Split realizado: {len(df_train)} train / {len(df_test)} test "
          f"(80/20 estratificado por dx, random_state={RANDOM_STATE}).")
    return df_train.reset_index(drop=True), df_test.reset_index(drop=True), mapeo_imagenes


# ==============================================================================
# 2. ARQUITECTURAS
# ==============================================================================

class DenseNetPlano(nn.Module):
    """
    [TFM] 5.4.4 / 5.5.4 intentos 1 y 2: "configuración plana tradicional".
    GAP de DenseNet201 -> una sola capa lineal de salida (7 clases).
    """

    def __init__(self, num_clases=NUM_CLASES):
        super().__init__()
        pesos = models.DenseNet201_Weights.DEFAULT
        self.densenet = models.densenet201(weights=pesos)
        num_features = self.densenet.classifier.in_features
        self.densenet.classifier = nn.Linear(num_features, num_clases)

    def forward(self, x):
        return self.densenet(x)


class DenseNetMultitarea(nn.Module):
    """
    [TFM] 5.5.4 intentos 3 y 4: arquitectura bifurcada en dos cabezas
    lineales paralelas sobre el mismo backbone DenseNet201.
      - cabeza_binaria: Cabeza A (triaje binario, 2 salidas)
      - cabeza_especifica: Cabeza B (diagnóstico específico, 7 salidas)
    Implementación replicada de TFM_1_v4.ipynb (clase DenseNetMultitarea),
    consistente con la descripción textual del TFM.
    """

    def __init__(self, num_clases_especificas=NUM_CLASES):
        super().__init__()
        pesos = models.DenseNet201_Weights.DEFAULT
        self.densenet = models.densenet201(weights=pesos)
        num_features = self.densenet.classifier.in_features
        self.densenet.classifier = nn.Identity()
        self.cabeza_binaria = nn.Linear(num_features, 2)
        self.cabeza_especifica = nn.Linear(num_features, num_clases_especificas)

    def forward(self, x):
        caracteristicas = self.densenet(x)
        salida_binaria = self.cabeza_binaria(caracteristicas)
        salida_especifica = self.cabeza_especifica(caracteristicas)
        return salida_binaria, salida_especifica


def generar_etiquetas_binarias(etiquetas_especificas):
    """Convierte etiquetas de 7 clases a 0=Benigno/1=Maligno. [TFM] 5.5.4."""
    es_maligno = torch.zeros_like(etiquetas_especificas, dtype=torch.bool)
    for idx in IDX_MALIGNAS:
        es_maligno |= (etiquetas_especificas == idx)
    return es_maligno.long()


def get_alpha_pesos(epoch_1indexed, experimento):
    """
    Devuelve (peso_binario, peso_especifico) para combinar las dos pérdidas.

    - "multitarea_fijo": 50/50 fijo. [TFM]
    - "multitarea_curriculum": función escalón [TFM, con interpretación
      [SUP] documentada en el docstring superior]:
          alpha=0.3 (épocas 1-15)  -> peso_especifico=0.3, peso_binario=0.7
          alpha=0.7 (épocas 16-30) -> peso_especifico=0.7, peso_binario=0.3
    """
    if experimento == "multitarea_fijo":
        return 0.5, 0.5
    if experimento == "multitarea_curriculum":
        alpha = 0.3 if epoch_1indexed <= 15 else 0.7  # [TFM]
        return (1 - alpha), alpha
    raise ValueError(f"get_alpha_pesos no aplica al experimento {experimento}")


def calcular_pesos_clase(df_train):
    """[TFM] 'vector de pesos ponderados ... Weighted Cross-Entropy Loss'."""
    conteo = df_train["dx"].value_counts().to_dict()
    muestras_por_clase = [conteo.get(c, 0) for c in NOMBRES_CLASES]
    total = sum(muestras_por_clase)
    pesos = [total / (NUM_CLASES * n) if n > 0 else 0.0 for n in muestras_por_clase]
    return torch.FloatTensor(pesos)


# ==============================================================================
# 3. ENTRENAMIENTO
# ==============================================================================

def _ruta_checkpoint(experimento):
    return os.path.join(RUTA_RESULTADOS, f"checkpoint_{experimento}.pt")


def _guardar_checkpoint(ruta, epoch, modelo, optimizer, scheduler, historial,
                         mejor_val_acc, mejor_epoca, mejor_estado, tiempo_acumulado_s):
    """
    Guarda el estado completo de entrenamiento tras completar `epoch`.
    Escribe primero a un archivo temporal y luego lo renombra (os.replace),
    para que una interrupcion a mitad del guardado no deje un checkpoint
    corrupto que no se pueda volver a cargar.
    """
    estado = {
        "epoch_completada": epoch,
        "model_state": modelo.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "historial": historial,
        "mejor_val_acc": mejor_val_acc,
        "mejor_epoca": mejor_epoca,
        "mejor_estado": mejor_estado,
        "tiempo_acumulado_s": tiempo_acumulado_s,
    }
    ruta_tmp = ruta + ".tmp"
    torch.save(estado, ruta_tmp)
    os.replace(ruta_tmp, ruta)


def entrenar(experimento, train_loader, val_loader, pesos_clase):
    """
    Entrena el modelo correspondiente al experimento elegido y devuelve
    (modelo, historial, mejor_epoca, tiempo_entrenamiento_s).

    [SUP] Guarda un checkpoint en disco (RUTA_RESULTADOS/checkpoint_<experimento>.pt)
    al terminar CADA EPOCA. Si el script se interrumpe (cierre de la terminal,
    apagon, Ctrl+C) y se vuelve a ejecutar, detecta el checkpoint y continua
    desde la siguiente epoca en lugar de reiniciar el entrenamiento completo.
    Esto es una medida practica para entrenamientos largos en CPU, no algo
    descrito en el TFM.
    """
    epocas = EPOCAS_POR_EXPERIMENTO[experimento]
    es_multitarea = experimento in ("multitarea_fijo", "multitarea_curriculum")
    es_con_da = experimento != "plano_sin_da"

    modelo = (DenseNetMultitarea() if es_multitarea else DenseNetPlano()).to(DEVICE)

    criterion_especifico = nn.CrossEntropyLoss(weight=pesos_clase.to(DEVICE))  # [TFM]
    criterion_binario = nn.CrossEntropyLoss()  # [SUP] sin ponderar; no se menciona
                                                # ponderacion para la Cabeza A en el TFM.

    optimizer = optim.Adam(modelo.parameters(), lr=OPTIMIZER_LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=SCHEDULER_FACTOR, patience=SCHEDULER_PATIENCE
    )  # [NB]

    historial = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "lr": []}
    mejor_val_acc = -1.0
    mejor_epoca = -1
    mejor_estado = None
    epoca_inicio = 1
    tiempo_acumulado_previo_s = 0.0

    ruta_ckpt = _ruta_checkpoint(experimento)
    if os.path.isfile(ruta_ckpt):
        print(f"\nSe encontro un checkpoint previo para '{experimento}' en {ruta_ckpt}.")
        ckpt = torch.load(ruta_ckpt, map_location=DEVICE, weights_only=False)
        modelo.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        scheduler.load_state_dict(ckpt["scheduler_state"])
        historial = ckpt["historial"]
        mejor_val_acc = ckpt["mejor_val_acc"]
        mejor_epoca = ckpt["mejor_epoca"]
        mejor_estado = ckpt["mejor_estado"]
        tiempo_acumulado_previo_s = ckpt.get("tiempo_acumulado_s", 0.0)
        epoca_inicio = ckpt["epoch_completada"] + 1
        if epoca_inicio > epocas:
            print(f"El checkpoint ya tiene las {epocas} epocas completadas. "
                  f"No se reentrena; se usan los pesos guardados.")
        else:
            print(f"Reanudando desde la epoca {epoca_inicio} de {epocas} "
                  f"(tiempo ya acumulado: {tiempo_acumulado_previo_s / 60:.1f} min).")

    print(f"\n=== Entrenando experimento '{experimento}' ({epocas} épocas, "
          f"{'con' if es_con_da else 'sin'} Data Augmentation, "
          f"{'doble cabeza' if es_multitarea else 'cabeza única'}) en {DEVICE} ===\n")

    t_inicio = time.time()

    for epoch in range(epoca_inicio, epocas + 1):
        t_epoch = time.time()
        modelo.train()
        running_loss, correct, total = 0.0, 0, 0

        for imagenes, etiquetas in train_loader:
            imagenes = imagenes.to(DEVICE)
            etiquetas = etiquetas.to(DEVICE)
            optimizer.zero_grad()

            if es_multitarea:
                peso_bin, peso_esp = get_alpha_pesos(epoch, experimento)
                etiquetas_bin = generar_etiquetas_binarias(etiquetas)
                pred_bin, pred_esp = modelo(imagenes)
                loss_b = criterion_binario(pred_bin, etiquetas_bin)
                loss_e = criterion_especifico(pred_esp, etiquetas)
                loss_total = peso_bin * loss_b + peso_esp * loss_e
                pred_final = pred_esp
            else:
                pred_final = modelo(imagenes)
                loss_total = criterion_especifico(pred_final, etiquetas)

            loss_total.backward()
            optimizer.step()

            running_loss += loss_total.item() * imagenes.size(0)
            _, predicted = torch.max(pred_final, 1)
            total += etiquetas.size(0)
            correct += (predicted == etiquetas).sum().item()

        train_loss = running_loss / total
        train_acc = correct / total

        modelo.eval()
        running_val_loss, correct_val, total_val = 0.0, 0, 0
        with torch.no_grad():
            for imagenes, etiquetas in val_loader:
                imagenes = imagenes.to(DEVICE)
                etiquetas = etiquetas.to(DEVICE)

                if es_multitarea:
                    peso_bin, peso_esp = get_alpha_pesos(epoch, experimento)
                    etiquetas_bin = generar_etiquetas_binarias(etiquetas)
                    pred_bin, pred_esp = modelo(imagenes)
                    loss_b = criterion_binario(pred_bin, etiquetas_bin)
                    loss_e = criterion_especifico(pred_esp, etiquetas)
                    loss_total = peso_bin * loss_b + peso_esp * loss_e
                    pred_final = pred_esp
                else:
                    pred_final = modelo(imagenes)
                    loss_total = criterion_especifico(pred_final, etiquetas)

                running_val_loss += loss_total.item() * imagenes.size(0)
                _, predicted = torch.max(pred_final, 1)
                total_val += etiquetas.size(0)
                correct_val += (predicted == etiquetas).sum().item()

        val_loss = running_val_loss / total_val
        val_acc = correct_val / total_val
        scheduler.step(val_loss)

        historial["train_loss"].append(train_loss)
        historial["val_loss"].append(val_loss)
        historial["train_acc"].append(train_acc)
        historial["val_acc"].append(val_acc)
        historial["lr"].append(optimizer.param_groups[0]["lr"])

        if val_acc > mejor_val_acc:
            mejor_val_acc = val_acc
            mejor_epoca = epoch
            mejor_estado = copy.deepcopy(modelo.state_dict())

        print(f"Época [{epoch}/{epocas}] | {time.time() - t_epoch:.1f}s | "
              f"Train loss {train_loss:.4f} acc {train_acc:.4f} | "
              f"Val loss {val_loss:.4f} acc {val_acc:.4f}")

        tiempo_hasta_ahora_s = tiempo_acumulado_previo_s + (time.time() - t_inicio)
        _guardar_checkpoint(ruta_ckpt, epoch, modelo, optimizer, scheduler, historial,
                             mejor_val_acc, mejor_epoca, mejor_estado, tiempo_hasta_ahora_s)

    tiempo_entrenamiento_s = tiempo_acumulado_previo_s + (time.time() - t_inicio)

    if mejor_estado is not None:
        modelo.load_state_dict(mejor_estado)
        print(f"\nMejor época según accuracy de validación: {mejor_epoca} "
              f"(val_acc={mejor_val_acc:.4f}). Se restauran esos pesos.")

    # Entrenamiento completo: el checkpoint ya no hace falta (el modelo final
    # se guarda por separado en main()). Se borra para no dejar archivos
    # duplicados ni confundir una corrida futura con una a medias.
    if os.path.isfile(ruta_ckpt):
        os.remove(ruta_ckpt)

    return modelo, historial, mejor_epoca, tiempo_entrenamiento_s


# ==============================================================================
# 4. EVALUACIÓN Y MÉTRICAS
# ==============================================================================

def evaluar_modelo(modelo, loader, experimento):
    """
    Evalúa el modelo final sobre el loader dado. Devuelve etiquetas reales,
    predicciones de Cabeza B (o única cabeza), probabilidades softmax y,
    si aplica, predicciones/etiquetas de la Cabeza A binaria.
    """
    es_multitarea = experimento in ("multitarea_fijo", "multitarea_curriculum")
    modelo.eval()

    y_true, y_pred, y_proba = [], [], []
    y_true_bin, y_pred_bin = [], []

    t_inicio_infer = time.time()
    n_imagenes = 0

    with torch.no_grad():
        for imagenes, etiquetas in loader:
            imagenes = imagenes.to(DEVICE)
            n_imagenes += imagenes.size(0)

            if es_multitarea:
                pred_bin, pred_esp = modelo(imagenes)
                probs_bin = torch.softmax(pred_bin, dim=1)
                _, pred_bin_clase = torch.max(pred_bin, 1)
                etiquetas_bin = generar_etiquetas_binarias(etiquetas)
                y_true_bin.extend(etiquetas_bin.numpy().tolist())
                y_pred_bin.extend(pred_bin_clase.cpu().numpy().tolist())
            else:
                pred_esp = modelo(imagenes)

            probs = torch.softmax(pred_esp, dim=1)
            _, pred_clase = torch.max(pred_esp, 1)

            y_true.extend(etiquetas.numpy().tolist())
            y_pred.extend(pred_clase.cpu().numpy().tolist())
            y_proba.extend(probs.cpu().numpy().tolist())

    tiempo_infer_total_s = time.time() - t_inicio_infer
    tiempo_infer_por_img_ms = (tiempo_infer_total_s / n_imagenes) * 1000 if n_imagenes else 0.0

    resultado = {
        "y_true": np.array(y_true),
        "y_pred": np.array(y_pred),
        "y_proba": np.array(y_proba),
        "tiempo_infer_total_s": tiempo_infer_total_s,
        "tiempo_infer_por_img_ms": tiempo_infer_por_img_ms,
    }
    if es_multitarea:
        resultado["y_true_bin"] = np.array(y_true_bin)
        resultado["y_pred_bin"] = np.array(y_pred_bin)
    return resultado


def calcular_metricas_globales(y_true, y_pred, y_proba):
    """
    Calcula el mismo conjunto de 13 métricas globales que reporta el SVM
    (Tabla 9 del TFM) para que la comparación sea directa.
    """
    y_true_bin = label_binarize(y_true, classes=list(range(NUM_CLASES)))

    try:
        auc_macro = roc_auc_score(y_true_bin, y_proba, average="macro", multi_class="ovr")
        auc_weighted = roc_auc_score(y_true_bin, y_proba, average="weighted", multi_class="ovr")
    except ValueError as e:
        print(f"AVISO: no se pudo calcular AUC ({e}). Se reporta como NaN.")
        auc_macro, auc_weighted = float("nan"), float("nan")

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_micro": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "auc_macro": auc_macro,
        "auc_weighted": auc_weighted,
        "cohen_kappa": cohen_kappa_score(y_true, y_pred),
        "mcc": matthews_corrcoef(y_true, y_pred),
    }


def calcular_metricas_por_clase(y_true, y_pred, y_proba):
    """Precision, recall, f1, soporte, AUC y average precision por clase."""
    y_true_bin = label_binarize(y_true, classes=list(range(NUM_CLASES)))
    precision, recall, f1, soporte = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(NUM_CLASES)), zero_division=0
    )
    filas = []
    for i, clase in enumerate(NOMBRES_CLASES):
        try:
            auc_clase = roc_auc_score(y_true_bin[:, i], y_proba[:, i])
            ap_clase = average_precision_score(y_true_bin[:, i], y_proba[:, i])
        except ValueError:
            auc_clase, ap_clase = float("nan"), float("nan")
        filas.append({
            "clase": clase, "soporte": int(soporte[i]),
            "precision": precision[i], "recall": recall[i], "f1": f1[i],
            "auc": auc_clase, "avg_precision": ap_clase,
        })
    return filas


def reconstruir_binario_desde_especifica(y_true, y_pred):
    """
    Agrega las predicciones de 7 clases a Benigno/Maligno, exactamente
    como hizo el TFM (sección 5.5.6.1, Tabla 17) para comparar la línea
    base SVM (multiclase) contra el modelo balanceado (binario nativo).
    Se aplica el mismo criterio aquí para comparar contra el SVM y, de
    paso, contra la Cabeza A nativa del propio modelo multitarea.
    """
    es_maligno_true = np.isin(y_true, IDX_MALIGNAS).astype(int)
    es_maligno_pred = np.isin(y_pred, IDX_MALIGNAS).astype(int)
    cm = confusion_matrix(es_maligno_true, es_maligno_pred, labels=[0, 1])
    recall_maligno = recall_score(es_maligno_true, es_maligno_pred, pos_label=1, zero_division=0)
    precision_maligno = precision_score(es_maligno_true, es_maligno_pred, pos_label=1, zero_division=0)
    f1_maligno = f1_score(es_maligno_true, es_maligno_pred, pos_label=1, zero_division=0)
    recall_benigno = recall_score(es_maligno_true, es_maligno_pred, pos_label=0, zero_division=0)
    return {
        "matriz_confusion": cm.tolist(),
        "recall_maligno": recall_maligno, "precision_maligno": precision_maligno,
        "f1_maligno": f1_maligno, "recall_benigno_especificidad": recall_benigno,
    }


# ==============================================================================
# 5. EXPORTACIÓN A EXCEL
# ==============================================================================

ENCABEZADO_FILL = PatternFill(start_color="0098CD", end_color="0098CD", fill_type="solid")
ENCABEZADO_FONT = Font(color="FFFFFF", bold=True)


def _formatear_encabezados(ws, fila=1):
    for cell in ws[fila]:
        if cell.value is not None:
            cell.fill = ENCABEZADO_FILL
            cell.font = ENCABEZADO_FONT
            cell.alignment = Alignment(horizontal="center")


def _autoancho(ws):
    for col_cells in ws.columns:
        max_len = max((len(str(c.value)) for c in col_cells if c.value is not None), default=8)
        ws.column_dimensions[get_column_letter(col_cells[0].column)].width = min(max_len + 2, 45)


def exportar_excel(ruta_salida, experimento, metricas_globales, metricas_por_clase,
                    cm, historial, mejor_epoca, tiempos, binario_agregado,
                    cabeza_a_metricas, n_train, n_test):
    wb = openpyxl.Workbook()

    # --- Hoja 1: RESUMEN -----------------------------------------------------
    ws = wb.active
    ws.title = "RESUMEN"
    ws.append(["Métrica", "Valor", "Valor (%)"])
    for clave, valor in metricas_globales.items():
        ws.append([clave, round(valor, 4), f"{valor * 100:.2f}%" if valor == valor else "N/D"])
    ws.append([])
    ws.append(["Experimento ejecutado", experimento])
    ws.append(["Mejor época (según val_acc)", mejor_epoca])
    ws.append(["Imágenes train", n_train])
    ws.append(["Imágenes test", n_test])
    _formatear_encabezados(ws)
    _autoancho(ws)

    # --- Hoja 2: MÉTRICAS_POR_CLASE ------------------------------------------
    ws2 = wb.create_sheet("MÉTRICAS_POR_CLASE")
    ws2.append(["Clase", "Soporte", "Precision", "Recall", "F1", "AUC", "Avg.Precision"])
    for fila in metricas_por_clase:
        ws2.append([fila["clase"], fila["soporte"], round(fila["precision"], 4),
                    round(fila["recall"], 4), round(fila["f1"], 4),
                    round(fila["auc"], 4) if fila["auc"] == fila["auc"] else "N/D",
                    round(fila["avg_precision"], 4) if fila["avg_precision"] == fila["avg_precision"] else "N/D"])
    _formatear_encabezados(ws2)
    _autoancho(ws2)

    # --- Hoja 3: MATRIZ_CONFUSIÓN --------------------------------------------
    ws3 = wb.create_sheet("MATRIZ_CONFUSIÓN")
    ws3.append([""] + [f"Pred: {c}" for c in NOMBRES_CLASES])
    for i, clase in enumerate(NOMBRES_CLASES):
        ws3.append([f"Real: {clase}"] + list(cm[i]))
    _formatear_encabezados(ws3)
    _autoancho(ws3)

    # --- Hoja 4: HISTORIAL_ENTRENAMIENTO -------------------------------------
    ws4 = wb.create_sheet("HISTORIAL_ENTRENAMIENTO")
    ws4.append(["Época", "Train Loss", "Val Loss", "Train Acc", "Val Acc", "LR"])
    for i in range(len(historial["train_loss"])):
        ws4.append([i + 1, round(historial["train_loss"][i], 4), round(historial["val_loss"][i], 4),
                    round(historial["train_acc"][i], 4), round(historial["val_acc"][i], 4),
                    historial["lr"][i]])
    _formatear_encabezados(ws4)
    _autoancho(ws4)

    # --- Hoja 5: CABEZA_BINARIA_Y_AGREGADO -----------------------------------
    ws5 = wb.create_sheet("CABEZA_BINARIA_Y_AGREGADO")
    ws5.append(["Vista", "Recall Maligno", "Precision Maligno", "F1 Maligno", "Recall Benigno (especificidad)"])
    ws5.append(["Agregado desde Cabeza B (7 clases -> binario)",
                round(binario_agregado["recall_maligno"], 4),
                round(binario_agregado["precision_maligno"], 4),
                round(binario_agregado["f1_maligno"], 4),
                round(binario_agregado["recall_benigno_especificidad"], 4)])
    if cabeza_a_metricas is not None:
        ws5.append(["Cabeza A nativa (triaje binario)",
                    round(cabeza_a_metricas["recall_maligno"], 4),
                    round(cabeza_a_metricas["precision_maligno"], 4),
                    round(cabeza_a_metricas["f1_maligno"], 4),
                    round(cabeza_a_metricas["recall_benigno_especificidad"], 4)])
        ws5.append([])
        ws5.append(["NOTA: la Cabeza A nativa se evalúa aquí de forma adicional;"
                    " el TFM (5.5.4) describe la arquitectura pero los notebooks"
                    " de referencia solo evalúan la Cabeza B. Se incluye porque"
                    " el propio TFM (5.5.6.2) señala como tarea pendiente poder"
                    " reconstruir este agregado para el Escenario 2; aquí se deja"
                    " calculado para el Escenario 1."])
    else:
        ws5.append(["Cabeza A nativa", "No aplica (experimento de cabeza única)"])
    _formatear_encabezados(ws5)
    _autoancho(ws5)

    # --- Hoja 6: CONFIGURACIÓN ------------------------------------------------
    ws6 = wb.create_sheet("CONFIGURACIÓN")
    config_rows = [
        ("Experimento", experimento),
        ("Backbone", "DenseNet201 (ImageNet, transfer learning) [TFM]"),
        ("Arquitectura", "Doble cabeza (binaria + específica)" if experimento.startswith("multitarea") else "Cabeza única (7 clases)"),
        ("Tamaño de imagen de entrada", f"{IMG_SIZE[0]}x{IMG_SIZE[1]} [NB - no especificado en el TFM]"),
        ("Batch size", f"{BATCH_SIZE} [NB - no especificado en el TFM]"),
        ("Épocas", EPOCAS_POR_EXPERIMENTO[experimento]),
        ("Optimizador", f"Adam, lr={OPTIMIZER_LR} [TFM], weight_decay={WEIGHT_DECAY} [NB]"),
        ("Scheduler", f"ReduceLROnPlateau factor={SCHEDULER_FACTOR} patience={SCHEDULER_PATIENCE} "
                       f"[NB - el TFM solo dice 'planificador dinámico' sin detallar tipo]"),
        ("Data Augmentation", "RandomResizedCrop, flips H/V, rotación ±20° [TFM]" if experimento != "plano_sin_da" else "Ninguno [TFM]"),
        ("Pérdida (cabeza específica)", "CrossEntropyLoss ponderada por frecuencia inversa de clase [TFM]"),
        ("Pérdida (cabeza binaria)", "CrossEntropyLoss estándar (sin ponderar) [SUP]" if experimento.startswith("multitarea") else "N/A"),
        ("Combinación de pérdidas", {"multitarea_fijo": "50% / 50% fijo [TFM]",
                                      "multitarea_curriculum": "Función escalón: alpha=0.3 (épocas 1-15), alpha=0.7 (épocas 16-30) [TFM, asignación de pesos SUP]"
                                      }.get(experimento, "N/A")),
        ("División train/test", f"80/20 estratificada por 'dx', random_state={RANDOM_STATE} [SUP - ver docstring]"),
        ("Semilla aleatoria", RANDOM_STATE),
        ("Dispositivo de entrenamiento", str(DEVICE)),
        ("Tiempo total de entrenamiento (s)", round(tiempos["train_s"], 3)),
        ("Tiempo total de inferencia en test (s)", round(tiempos["infer_total_s"], 3)),
        ("Tiempo de inferencia por imagen (ms)", round(tiempos["infer_por_img_ms"], 4)),
        ("Fecha de ejecución", datetime.now().strftime("%Y-%m-%d %H:%M:%S")),
    ]
    ws6.append(["Parámetro", "Valor"])
    for k, v in config_rows:
        ws6.append([k, v])
    _formatear_encabezados(ws6)
    _autoancho(ws6)

    # --- Hoja 7: COMPARACIÓN_SVM_vs_DENSENET ---------------------------------
    ws7 = wb.create_sheet("COMPARACIÓN_SVM_vs_DENSENET")
    ws7.append(["Métrica", "SVM Escenario 1 (TFM, Tabla 9)", f"DenseNet201 Escenario 1 ({experimento})"])
    for clave in SVM_ESCENARIO1_GLOBAL:
        valor_dn = metricas_globales.get(clave, float("nan"))
        ws7.append([clave, round(SVM_ESCENARIO1_GLOBAL[clave], 4),
                    round(valor_dn, 4) if valor_dn == valor_dn else "N/D"])
    ws7.append([])
    ws7.append(["Recall clase Maligna agregada (akiec+bcc+mel)",
                "0.6450 [TFM, Tabla 18: 252/391 -- nota: ese valor corresponde al test"
                " set específico de la comparación 5.5.6.1, puede diferir levemente"
                " del recall agregado recalculado aquí sobre el split de este script]",
                round(binario_agregado["recall_maligno"], 4)])
    ws7.append([])
    ws7.append(["Tiempo de entrenamiento (s)", round(SVM_ESCENARIO1_TIEMPOS["train_s"], 3),
                round(tiempos["train_s"], 3)])
    ws7.append(["Tiempo de inferencia por imagen (ms)", round(SVM_ESCENARIO1_TIEMPOS["infer_per_img_ms"], 4),
                round(tiempos["infer_por_img_ms"], 4)])
    ws7.append([])
    ws7.append(["NOTA METODOLÓGICA", "El TFM (5.5.6.3) advierte que no reporta tiempos de"
                " entrenamiento/inferencia para DenseNet201 en ningún escenario, dejándolo"
                " como tarea pendiente. Este script mide ambos directamente, por lo que"
                " estas dos últimas filas SÍ son datos nuevos, no tomados del documento."])
    _formatear_encabezados(ws7)
    _autoancho(ws7)

    wb.save(ruta_salida)
    print(f"\nExcel de resultados guardado en: {ruta_salida}")


# ==============================================================================
# 6. MAIN
# ==============================================================================

def main():
    df_train, df_test, mapeo_imagenes = cargar_metadata_y_split()

    transform_train = TRANSFORM_SIN_DA if EXPERIMENTO == "plano_sin_da" else TRANSFORM_TRAIN
    transform_eval = TRANSFORM_SIN_DA if EXPERIMENTO == "plano_sin_da" else TRANSFORM_EVAL

    ds_train = HAM10000Dataset(df_train, mapeo_imagenes, transform_train)
    ds_test = HAM10000Dataset(df_test, mapeo_imagenes, transform_eval)

    train_loader = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    test_loader = DataLoader(ds_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    pesos_clase = calcular_pesos_clase(df_train)

    try:
        modelo, historial, mejor_epoca, tiempo_entrenamiento_s = entrenar(
            EXPERIMENTO, train_loader, test_loader, pesos_clase
        )
    except KeyboardInterrupt:
        print(f"\n\nEntrenamiento interrumpido manualmente (Ctrl+C). "
              f"El progreso hasta la última época completada quedó guardado en:\n"
              f"  {_ruta_checkpoint(EXPERIMENTO)}\n"
              f"Vuelve a ejecutar el script (con el mismo EXPERIMENTO = '{EXPERIMENTO}') "
              f"para continuar desde donde se quedó.")
        return

    resultado_eval = evaluar_modelo(modelo, test_loader, EXPERIMENTO)
    y_true, y_pred, y_proba = resultado_eval["y_true"], resultado_eval["y_pred"], resultado_eval["y_proba"]

    metricas_globales = calcular_metricas_globales(y_true, y_pred, y_proba)
    metricas_por_clase = calcular_metricas_por_clase(y_true, y_pred, y_proba)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASES)))
    binario_agregado = reconstruir_binario_desde_especifica(y_true, y_pred)

    cabeza_a_metricas = None
    if "y_true_bin" in resultado_eval:
        y_true_bin_real = resultado_eval["y_true_bin"]
        y_pred_bin_real = resultado_eval["y_pred_bin"]
        cabeza_a_metricas = {
            "recall_maligno": recall_score(y_true_bin_real, y_pred_bin_real, pos_label=1, zero_division=0),
            "precision_maligno": precision_score(y_true_bin_real, y_pred_bin_real, pos_label=1, zero_division=0),
            "f1_maligno": f1_score(y_true_bin_real, y_pred_bin_real, pos_label=1, zero_division=0),
            "recall_benigno_especificidad": recall_score(y_true_bin_real, y_pred_bin_real, pos_label=0, zero_division=0),
        }

    tiempos = {
        "train_s": tiempo_entrenamiento_s,
        "infer_total_s": resultado_eval["tiempo_infer_total_s"],
        "infer_por_img_ms": resultado_eval["tiempo_infer_por_img_ms"],
    }

    print("\n=== MÉTRICAS GLOBALES (test) ===")
    for k, v in metricas_globales.items():
        print(f"  {k}: {v:.4f}" if v == v else f"  {k}: N/D")

    # --- Guardar modelo y JSON (espejo de svm_model.joblib / metricas_baseline.json)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    nombre_base = f"densenet201_ham10000_{EXPERIMENTO}_{timestamp}"

    ruta_modelo = os.path.join(RUTA_RESULTADOS, f"{nombre_base}.pth")
    torch.save(modelo.state_dict(), ruta_modelo)
    print(f"Modelo guardado en: {ruta_modelo}")

    metricas_json = {
        "run_id": timestamp,
        "experimento": EXPERIMENTO,
        "dataset_info": {
            "n_imagenes_total": len(df_train) + len(df_test),
            "n_train": len(df_train), "n_test": len(df_test),
            "n_clases": NUM_CLASES, "img_size": IMG_SIZE, "batch_size": BATCH_SIZE,
        },
        "global_metrics": metricas_globales,
        "per_class_metrics": metricas_por_clase,
        "binario_agregado_desde_especifica": binario_agregado,
        "cabeza_a_nativa": cabeza_a_metricas,
        "timing_seconds": tiempos,
        "mejor_epoca": mejor_epoca,
        "model_config": {
            "optimizer": "Adam", "lr": OPTIMIZER_LR, "weight_decay": WEIGHT_DECAY,
            "scheduler": "ReduceLROnPlateau", "scheduler_factor": SCHEDULER_FACTOR,
            "scheduler_patience": SCHEDULER_PATIENCE,
        },
    }
    ruta_json = os.path.join(RUTA_RESULTADOS, f"{nombre_base}.json")
    with open(ruta_json, "w", encoding="utf-8") as f:
        json.dump(metricas_json, f, indent=2, ensure_ascii=False)
    print(f"Métricas JSON guardadas en: {ruta_json}")

    ruta_excel = os.path.join(RUTA_RESULTADOS, f"{nombre_base}.xlsx")
    exportar_excel(ruta_excel, EXPERIMENTO, metricas_globales, metricas_por_clase, cm,
                   historial, mejor_epoca, tiempos, binario_agregado, cabeza_a_metricas,
                   len(df_train), len(df_test))


if __name__ == "__main__":
    main()


Imágenes encontradas en disco (búsqueda recursiva bajo /content/HAM10000): 10015. Filas del CSV emparejadas con una imagen: 10015 de 10015 (100.0%).
Split realizado: 8012 train / 2003 test (80/20 estratificado por dx, random_state=42).
Downloading: "https://download.pytorch.org/models/densenet201-c1103571.pth" to /root/.cache/torch/hub/checkpoints/densenet201-c1103571.pth


100%|██████████| 77.4M/77.4M [00:00<00:00, 198MB/s]



=== Entrenando experimento 'multitarea_curriculum' (30 épocas, con Data Augmentation, doble cabeza) en cuda ===

Época [1/30] | 229.4s | Train loss 0.5412 acc 0.6338 | Val loss 0.4103 acc 0.7044
Época [2/30] | 231.6s | Train loss 0.3905 acc 0.7346 | Val loss 0.3610 acc 0.7728
Época [3/30] | 231.3s | Train loss 0.3258 acc 0.7777 | Val loss 0.3442 acc 0.7798
Época [4/30] | 231.0s | Train loss 0.2868 acc 0.7988 | Val loss 0.3314 acc 0.8148
Época [5/30] | 231.3s | Train loss 0.2512 acc 0.8273 | Val loss 0.3180 acc 0.8313
Época [6/30] | 229.7s | Train loss 0.2176 acc 0.8533 | Val loss 0.3400 acc 0.8333
Época [7/30] | 229.4s | Train loss 0.1849 acc 0.8616 | Val loss 0.3420 acc 0.8402
Época [8/30] | 230.2s | Train loss 0.1600 acc 0.8779 | Val loss 0.3287 acc 0.8507
Época [9/30] | 230.1s | Train loss 0.1643 acc 0.8879 | Val loss 0.2857 acc 0.8462
Época [10/30] | 229.4s | Train loss 0.1328 acc 0.8984 | Val loss 0.2971 acc 0.8472
Época [11/30] | 229.8s | Train loss 0.1210 acc 0.9119 | Val loss 